# Week 5 — Multimodal 문서 파싱 (UpstageLayoutAnalysisLoader)

**목표**: Upstage Document Parse API를 이용해 PDF를 **텍스트 / 표 / 이미지** 로 분리하고,  
각 요소의 위치(바운딩 박스)를 페이지 위에 시각화한다.

| 방법 | 표 처리 | 이미지 처리 | 섹션 맥락 |
|---|---|---|---|
| Week4 PyMuPDF | 구조 소실 | 누락 | breadcrumb 보존 |
| **Week5 Upstage** | **HTML 테이블 태그 보존** | **이미지 캡션 추출** | **레이아웃 분석 기반** |

> **선택 이유**: `UpstageLayoutAnalysisLoader`는 딥러닝 기반 레이아웃 분석으로  
> PyMuPDF의 규칙 기반 방식이 놓치는 복잡한 표 구조와 이미지를 요소 단위로 분리한다.

In [ ]:
import os
import fitz  # PyMuPDF — 페이지 렌더링용
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from collections import defaultdict
from dotenv import load_dotenv
from langchain_upstage import UpstageLayoutAnalysisLoader

load_dotenv('../.env')

for font in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        matplotlib.rc('font', family=font)
        break
    except Exception:
        pass
matplotlib.rcParams['axes.unicode_minus'] = False

PDF_PATH = '../data/registration_of_real_estatee_manual.pdf'
print(f"UPSTAGE_API_KEY 설정 여부: {'설정됨' if os.getenv('UPSTAGE_API_KEY') else '미설정'}")

## 1. UpstageLayoutAnalysisLoader로 PDF 파싱

### 파라미터 선택 이유
| 파라미터 | 선택값 | 이유 |
|---|---|---|
| `split` | `"element"` | 텍스트/표/이미지를 **개별 Document**로 분리해야 카테고리별 처리 가능 |
| `output_type` | `"html"` | 표(table)의 행·열 구조를 HTML 태그로 보존 (`text`는 평탄화됨) |
| `use_ocr` | `True` | PDF 내 이미지에 포함된 텍스트(스캔본)까지 추출 |

In [ ]:
loader = UpstageLayoutAnalysisLoader(
    file_path=PDF_PATH,
    split='element',    # 요소 단위 분리 (vs 'page' / 'none')
    output_type='html', # 표 구조 보존 (vs 'text')
    use_ocr=True,
)

docs = loader.load()
print(f'총 추출 요소 수: {len(docs)}')
print(f'\n--- 첫 번째 요소 샘플 ---')
print(f"category : {docs[0].metadata.get('category')}")
print(f"page     : {docs[0].metadata.get('page')}")
print(f"metadata keys: {list(docs[0].metadata.keys())}")
print(f"content[:300]:\n{docs[0].page_content[:300]}")

## 2. 텍스트 / 표 / 이미지 분리

In [ ]:
# 카테고리별 그룹화
elements_by_cat = defaultdict(list)
for doc in docs:
    cat = doc.metadata.get('category', 'unknown')
    elements_by_cat[cat].append(doc)

print('=' * 45)
print('카테고리별 요소 수')
print('=' * 45)
max_cnt = max(len(v) for v in elements_by_cat.values())
for cat, elems in sorted(elements_by_cat.items(), key=lambda x: -len(x[1])):
    bar = '█' * (len(elems) * 30 // max(1, max_cnt))
    print(f'  {cat:<20} {len(elems):>5}개  {bar}')
print('=' * 45)

# Upstage 카테고리 → 대분류 매핑
TEXT_CATS   = {'paragraph', 'list', 'caption', 'header', 'footer', 'title'}
TABLE_CATS  = {'table'}
FIGURE_CATS = {'figure'}

text_docs   = [d for d in docs if d.metadata.get('category') in TEXT_CATS]
table_docs  = [d for d in docs if d.metadata.get('category') in TABLE_CATS]
figure_docs = [d for d in docs if d.metadata.get('category') in FIGURE_CATS]

print(f'\n[분리 결과]')
print(f'  텍스트 요소 : {len(text_docs):>5}개  (paragraph/list/caption/header/footer/title)')
print(f'  표 요소     : {len(table_docs):>5}개')
print(f'  이미지 요소 : {len(figure_docs):>5}개')

## 3. 각 카테고리 샘플 출력

In [ ]:
print('=' * 60)
print('[ 텍스트 요소 샘플 (3개) ]')
print('=' * 60)
for d in text_docs[:3]:
    print(f"\n  page={d.metadata.get('page')}  category={d.metadata.get('category')}")
    print(f'  {d.page_content[:300]}')
    print('-' * 60)

In [ ]:
from IPython.display import display, HTML

print('=' * 60)
print(f'[ 표 요소 샘플 ({min(3, len(table_docs))}개) — HTML 구조 보존 확인 ]')
print('=' * 60)

for d in table_docs[:3]:
    print(f"\n  page={d.metadata.get('page')}  category={d.metadata.get('category')}")
    display(HTML(
        "<div style='border:2px solid #DD8452;padding:10px;margin:6px;"
        "border-radius:6px;font-size:12px;background:#fffaf5'>"
        + d.page_content[:1500]
        + "</div>"
    ))

In [ ]:
print('=' * 60)
print(f'[ 이미지 요소 샘플 ({min(3, len(figure_docs))}개) ]')
print('=' * 60)

for d in figure_docs[:3]:
    print(f"\n  page={d.metadata.get('page')}  category={d.metadata.get('category')}")
    print(f"  content : {d.page_content[:200]}")
    print(f"  coordinates: {d.metadata.get('coordinates')}")
    print('-' * 60)

## 4. 파싱 결과 시각화

### 4-1. 카테고리 분포 차트

In [ ]:
# 카테고리별 색상
CATEGORY_COLORS = {
    'paragraph': '#4C72B0',
    'table'    : '#DD8452',
    'figure'   : '#55A868',
    'list'     : '#C44E52',
    'caption'  : '#8172B2',
    'header'   : '#937860',
    'footer'   : '#DA8BC3',
    'title'    : '#8C8C8C',
    'equation' : '#CCB974',
}
def get_color(cat):
    return CATEGORY_COLORS.get(cat, '#AAAAAA')

cat_counts = {cat: len(elems) for cat, elems in
              sorted(elements_by_cat.items(), key=lambda x: -len(x[1]))}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Upstage Layout Analysis — 카테고리 분포', fontsize=13, fontweight='bold')

# 바 차트
ax1 = axes[0]
cats   = list(cat_counts.keys())
counts = list(cat_counts.values())
colors = [get_color(c) for c in cats]
bars = ax1.barh(cats[::-1], counts[::-1], color=colors[::-1], edgecolor='white')
for bar, cnt in zip(bars, counts[::-1]):
    ax1.text(bar.get_width() + max(counts)*0.01, bar.get_y() + bar.get_height()/2,
             str(cnt), va='center', fontsize=9)
ax1.set_xlabel('요소 수')
ax1.set_title('카테고리별 요소 수')
ax1.grid(axis='x', alpha=0.3)

# 파이 차트 (대분류)
ax2 = axes[1]
other_cnt = len(docs) - len(text_docs) - len(table_docs) - len(figure_docs)
major_labels = ['텍스트\n(paragraph/list/etc)', '표\n(table)', '이미지\n(figure)']
major_values = [len(text_docs), len(table_docs), len(figure_docs)]
major_colors = ['#4C72B0', '#DD8452', '#55A868']
if other_cnt > 0:
    major_labels.append('기타')
    major_values.append(other_cnt)
    major_colors.append('#AAAAAA')

wedges, texts, autotexts = ax2.pie(
    major_values, labels=major_labels, colors=major_colors,
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    textprops={'fontsize': 9}
)
ax2.set_title('대분류 비율')

plt.tight_layout()
plt.show()

### 4-2. 페이지별 바운딩 박스 시각화

PyMuPDF로 PDF 페이지를 이미지로 렌더링한 뒤,  
Upstage가 감지한 요소의 바운딩 박스를 **카테고리 색상**으로 오버레이한다.

In [ ]:
def get_bbox_pixels(coordinates, img_w, img_h):
    """Upstage 좌표(normalized 0-1) → 픽셀 (x1, y1, x2, y2) 변환"""
    if coordinates is None:
        return None
    # 형식 A: [{"x": ..., "y": ...}, ...] 4개 꼭짓점
    if isinstance(coordinates, list) and len(coordinates) >= 2:
        xs = [p['x'] for p in coordinates if 'x' in p]
        ys = [p['y'] for p in coordinates if 'y' in p]
        if not xs:
            return None
        return (min(xs)*img_w, min(ys)*img_h, max(xs)*img_w, max(ys)*img_h)
    # 형식 B: {"x1": ..., "y1": ..., "x2": ..., "y2": ...}
    if isinstance(coordinates, dict):
        x1 = coordinates.get('x1', coordinates.get('left',   0))
        y1 = coordinates.get('y1', coordinates.get('top',    0))
        x2 = coordinates.get('x2', coordinates.get('right',  1))
        y2 = coordinates.get('y2', coordinates.get('bottom', 1))
        return (x1*img_w, y1*img_h, x2*img_w, y2*img_h)
    return None


def render_page_with_boxes(pdf_path, page_docs_map, page_num, dpi=100):
    """PDF 페이지를 렌더링하고 요소 바운딩 박스를 오버레이"""
    pdf = fitz.open(pdf_path)
    mat = fitz.Matrix(dpi/72, dpi/72)
    pix = pdf[page_num - 1].get_pixmap(matrix=mat)
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    pdf.close()

    fig, ax = plt.subplots(figsize=(8, 11))
    ax.imshow(img)
    ax.set_title(f'Page {page_num} — Upstage 파싱 결과', fontsize=11)
    ax.axis('off')

    legend_patches = {}
    for doc in page_docs_map.get(page_num, []):
        cat   = doc.metadata.get('category', 'unknown')
        bbox  = get_bbox_pixels(doc.metadata.get('coordinates'), pix.width, pix.height)
        if bbox is None:
            continue
        x1, y1, x2, y2 = bbox
        color = get_color(cat)
        rect = mpatches.FancyBboxPatch(
            (x1, y1), x2-x1, y2-y1,
            boxstyle='square,pad=0', linewidth=2,
            edgecolor=color, facecolor=color, alpha=0.18,
        )
        ax.add_patch(rect)
        ax.text(x1+2, y1+12, cat, fontsize=6.5, color=color,
                fontweight='bold', clip_on=True)
        if cat not in legend_patches:
            legend_patches[cat] = mpatches.Patch(color=color, label=cat, alpha=0.7)

    if legend_patches:
        ax.legend(handles=list(legend_patches.values()),
                  loc='upper right', fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.show()


# 페이지별 doc 맵
page_docs_map = defaultdict(list)
for doc in docs:
    pg = doc.metadata.get('page')
    if pg is not None:
        page_docs_map[int(pg)].append(doc)

print(f'요소가 있는 페이지 수: {len(page_docs_map)}')

In [ ]:
pages_with_table  = sorted({int(d.metadata['page']) for d in table_docs  if d.metadata.get('page')})
pages_with_figure = sorted({int(d.metadata['page']) for d in figure_docs if d.metadata.get('page')})

print(f'표 포함 페이지 (상위 10): {pages_with_table[:10]}')
print(f'이미지 포함 페이지 (상위 10): {pages_with_figure[:10]}')

# 텍스트 전용 / 표 포함 / 이미지 포함 페이지 각 1개 시각화
sample_pages = []
for pg in sorted(page_docs_map.keys()):
    cats = {d.metadata.get('category') for d in page_docs_map[pg]}
    if not (cats & {'table', 'figure'}) and 'paragraph' in cats:
        sample_pages.append((pg, '텍스트 페이지'))
        break
if pages_with_table:
    sample_pages.append((pages_with_table[0], '표 포함 페이지'))
if pages_with_figure:
    sample_pages.append((pages_with_figure[0], '이미지 포함 페이지'))

for pg, label in sample_pages:
    print(f'\n▶ {label} (page {pg}) 시각화:')
    render_page_with_boxes(PDF_PATH, page_docs_map, pg)

### 4-3. 여러 페이지 한 번에 비교 — 그리드 시각화

In [ ]:
def render_page_array(pdf_path, page_num, dpi=72):
    """페이지 numpy array + 픽셀 크기 반환"""
    pdf = fitz.open(pdf_path)
    mat = fitz.Matrix(dpi/72, dpi/72)
    pix = pdf[page_num - 1].get_pixmap(matrix=mat)
    arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    pdf.close()
    return arr, pix.width, pix.height


# 표·이미지 혼합 페이지 최대 6개
mixed_pages = sorted(set(pages_with_table[:3]) | set(pages_with_figure[:3]))[:6]
if not mixed_pages:
    mixed_pages = sorted(page_docs_map.keys())[:6]

n_cols = 3
n_rows = (len(mixed_pages) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 6))
axes_flat = np.array(axes).flatten()

for idx, pg in enumerate(mixed_pages):
    ax = axes_flat[idx]
    arr, img_w, img_h = render_page_array(PDF_PATH, pg, dpi=80)
    ax.imshow(arr)

    page_cats = set()
    for doc in page_docs_map.get(pg, []):
        cat  = doc.metadata.get('category', 'unknown')
        bbox = get_bbox_pixels(doc.metadata.get('coordinates'), img_w, img_h)
        if bbox:
            x1, y1, x2, y2 = bbox
            color = get_color(cat)
            ax.add_patch(plt.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.2
            ))
            page_cats.add(cat)

    handles = [mpatches.Patch(color=get_color(c), label=c, alpha=0.8) for c in page_cats]
    if handles:
        ax.legend(handles=handles, loc='upper right', fontsize=6, framealpha=0.9)
    ax.set_title(f'Page {pg}', fontsize=9)
    ax.axis('off')

for ax in axes_flat[len(mixed_pages):]:
    ax.axis('off')

fig.suptitle('표·이미지 포함 페이지 — 바운딩 박스 그리드', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4-4. 페이지별 요소 구성 히트맵

문서 전체에서 각 카테고리가 어느 페이지에 집중되는지 한눈에 파악한다.

In [ ]:
all_cats = sorted(elements_by_cat.keys())
all_pgs  = sorted(page_docs_map.keys())

# 페이지 × 카테고리 카운트 행렬
matrix = np.zeros((len(all_cats), len(all_pgs)), dtype=int)
for j, pg in enumerate(all_pgs):
    for doc in page_docs_map[pg]:
        cat = doc.metadata.get('category', 'unknown')
        if cat in all_cats:
            matrix[all_cats.index(cat), j] += 1

# 페이지 수 > 50 이면 10페이지 단위 구간으로 묶어 표시
if len(all_pgs) > 50:
    bin_sz  = 10
    n_bins  = (len(all_pgs) + bin_sz - 1) // bin_sz
    binned  = np.zeros((len(all_cats), n_bins), dtype=int)
    x_labels = []
    for b in range(n_bins):
        s = b * bin_sz
        e = min(s + bin_sz, len(all_pgs))
        binned[:, b] = matrix[:, s:e].sum(axis=1)
        x_labels.append(f'{all_pgs[s]}~{all_pgs[e-1]}')
    plot_mat = binned
    xlabel   = '페이지 구간 (10페이지 단위)'
else:
    plot_mat = matrix
    x_labels = [str(p) for p in all_pgs]
    xlabel   = '페이지'

fig, ax = plt.subplots(figsize=(16, max(4, len(all_cats) * 0.7)))
im = ax.imshow(plot_mat, aspect='auto', cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=ax, label='요소 수')
ax.set_yticks(range(len(all_cats)))
ax.set_yticklabels(all_cats, fontsize=9)
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, fontsize=7, rotation=45, ha='right')
ax.set_xlabel(xlabel)
ax.set_ylabel('카테고리')
ax.set_title('페이지별 카테고리 요소 분포 히트맵', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. 요약 및 RAG 설계 시사점

| 구분 | PyMuPDF (Week4) | Upstage LayoutAnalysis (Week5) |
|---|---|---|
| **텍스트** | get_text() 평탄화 | paragraph/list/title 구분 |
| **표** | 셀 개행 평탄화, 구조 소실 | HTML 태그로 행·열 구조 보존 |
| **이미지** | 완전 누락 | figure 요소로 감지, 캡션 연결 가능 |
| **좌표** | 없음 | 바운딩 박스(normalized 0-1) 제공 |
| **비용** | 무료 (로컬) | API 호출 비용 발생 |

**RAG 청킹 전략**:
1. **텍스트** (`paragraph`, `list`): RecursiveCharacterTextSplitter (chunk_size=1000)
2. **표** (`table`): HTML 그대로 1개 청크 — 행·열 구조 보존, 별도 메타데이터 `type=table`
3. **이미지** (`figure`): 캡션을 텍스트로 저장, 필요시 Vision LLM으로 이미지 설명 생성 후 추가 청크

In [ ]:
print('=' * 55)
print('  Upstage Layout Analysis 파싱 결과 요약')
print('=' * 55)
print(f'  총 추출 요소         : {len(docs):>6}개')
print(f'  텍스트 요소          : {len(text_docs):>6}개')
print(f'  표 요소              : {len(table_docs):>6}개')
print(f'  이미지 요소          : {len(figure_docs):>6}개')
print(f'  표 포함 페이지       : {len(pages_with_table):>6}페이지  {pages_with_table[:5]}')
print(f'  이미지 포함 페이지   : {len(pages_with_figure):>6}페이지  {pages_with_figure[:5]}')
print()
if table_docs:
    sample = table_docs[0].page_content[:400]
    print('  [표 HTML 구조 보존 확인]')
    print(f'    <tr> 태그 포함: {"<tr" in sample.lower()}')
    print(f'    <td> 태그 포함: {"<td" in sample.lower()}')
    preserved = '<tr' in sample.lower() or '<td' in sample.lower()
    print(f'    → 표 구조가 HTML로 {"보존됨" if preserved else "평탄화됨"}')
print('=' * 55)